# Approximation of the area under the  $f(x) = \frac{ 4 }{ 1 + x^2 }$ function curve using rectangles

In [ ]:
def f(x):
	return 4 / (1 + x ** 2)

In [ ]:
import math
import numpy as np
import threading
import multiprocessing

In [ ]:
number_rectangles = 1000
x_lower_limit = 0
x_upper_limit = 1

## Python

### Sequential version

In [ ]:
def work(x_lower_limit, x_upper_limit, number_rectangles):
	"""
	Calculates the area under the curve using rectangles.

	Args:
	- x_lower_limit: lower limit of the x axis.
	- x_upper_limit: upper limit of the x axis.
	- number_rectangles: number of rectangles with
	which to calculate the area under the curve.
	- number_threads: number of threads to perform
	the calculation.
	- thread_number: identifier of the actual
	thread.
	"""
	rectangle_width = (x_upper_limit - x_lower_limit) / number_rectangles

	# Generate x values
	x_values =  [x_lower_limit + rectangle_width * i for i in range(number_rectangles)] # np.linspace(x_lower_limit, x_upper_limit - rectangle_width, number_rectangles)

	# Generate y values
	y_values = [f(x_value) for x_value in x_values] # f(x_values)

	rectangle_areas = []
	area = 0
	for rectangle_height in y_values:
		rectangle_area = rectangle_width * rectangle_height
		area += rectangle_area
		rectangle_areas.append(rectangle_area)

	return area

In [ ]:
print(f"Total area: {work(x_lower_limit, x_upper_limit, number_rectangles)}")

### `threading` version

In [ ]:
number_threads = 4
threads = []
sums = [0] * number_threads

In [ ]:
def work(x_lower_limit, x_upper_limit, number_rectangles,
		number_threads, thread_number):
	"""
	Calculates the area under the curve using rectangles.

	Args:
	- x_lower_limit: lower limit of the x axis.
	- x_upper_limit: upper limit of the x axis.
	- number_rectangles: number of rectangles with
	which to calculate the area under the curve.
	- number_threads: number of threads to perform
	the calculation.
	- thread_number: identifier of the actual
	thread.
	"""
	partial_area = 0
	width = (x_upper_limit - x_lower_limit) / number_rectangles
	rectangles_per_thread = math.ceil(number_rectangles / number_threads)

	"""
	start_index = rectangles_per_thread * thread_number
	end_index = min(rectangles_per_thread * (thread_number + 1), number_rectangles)
	step = 1
	"""
	start_index = thread_number
	end_index = number_rectangles
	step = number_threads

	for rectangle in range(start_index, end_index, step):
		x = x_lower_limit + width * rectangle
		height = f(x)
		partial_area += width * height

	sums[thread_number] = partial_area

In [ ]:
for thread_number in range(number_threads):
	thread = threading.Thread(target=work, args=(x_lower_limit, x_upper_limit,
												number_rectangles, number_threads,
												thread_number))
	threads.append(thread)
	thread.start()

for thread in threads:
	thread.join()

area = sum(sums)
print(f"Total area: {area}")

### `multiprocessing` version

In [ ]:
number_processes = 4
processes = []
sums = multiprocessing.Array("d", number_processes)

In [ ]:
def work(x_lower_limit, x_upper_limit, number_rectangles,
		number_processes, process_number):
	"""
	Calculates the area under the curve using rectangles.

	Args:
	- x_lower_limit: lower limit of the x axis.
	- x_upper_limit: upper limit of the x axis.
	- number_rectangles: number of rectangles with
	which to calculate the area under the curve.
	- number_processes: number of processes to perform
	the calculation.
	- process_number: identifier of the actual
	process.
	"""
	partial_area = 0
	width = (x_upper_limit - x_lower_limit) / number_rectangles
	rectangles_per_process = math.ceil(number_rectangles / number_processes)

	"""
	start_index = rectangles_per_process * process_number
	end_index = min(rectangles_per_process * (process_number + 1), number_rectangles)
	step = 1
	"""
	start_index = process_number
	end_index = number_rectangles
	step = number_processes

	for rectangle in range(start_index, end_index, step):
		x = x_lower_limit + width * rectangle
		height = f(x)
		partial_area += width * height

	sums[process_number] = partial_area

In [ ]:
for process_number in range(number_processes):
	process = multiprocessing.Process(target=work, args=(x_lower_limit, x_upper_limit,
														number_rectangles, number_processes,
														process_number))
	processes.append(process)
	process.start()

for process in processes:
	process.join()

area = sum(sums)
print(f"Total area: {area}")

Total area: 3.1425924869231263


## C

### Sequential version

In [13]:
!gcc sequential.c -o sequential &&  time ./sequential 1000000

Total area: 3.142646
real	0m0.009s
user	0m0.007s
sys	0m0.002s


### `pthread.h` version

In [25]:
!gcc -pthread pthread.c -o pthread  && time ./pthread 4 10000000

Total area: 3.133886
real	0m0.070s
user	0m0.127s
sys	0m0.002s


### OpenMP

In [38]:
!gcc -fopenmp openmp.c -o openmp  && time ./openmp 4 10000000

Total area: 3.133886
real	0m0.067s
user	0m0.118s
sys	0m0.000s
